# Calculate Commutator Norms for Quantum State Transfer Protocols 

In [ ]:
import numpy as np
from scipy.sparse import csc_matrix, kron, identity
import scipy.sparse
from scipy.sparse.linalg import expm, norm, eigs
from scipy.linalg import qr
import matplotlib.pyplot as plt

## Generate Pauli matrices

In [ ]:
sx = csc_matrix([[0,1],[1,0]])
sz = csc_matrix([[1,0],[0,-1]])
sy = csc_matrix([[0,-1j],[1j,0]])
I2 = csc_matrix([[1,0],[0,1]])

def generate_Paulis(L):
    X = []
    Z = []
    Y = []
    for i in range(L):
        X_i = sx.copy()
        Z_i = sz.copy()
        Y_i = sy.copy()
        for j in range(L):
            if j < i:
                X_i = kron(I2, X_i, format='csc')
                Z_i = kron(I2, Z_i, format='csc')
                Y_i = kron(I2, Y_i, format='csc')
            if j > i:
                X_i = kron(X_i, I2, format='csc')
                Z_i = kron(Z_i, I2, format='csc')
                Y_i = kron(Y_i, I2, format='csc')
        X.append(X_i)
        Z.append(Z_i)
        Y.append(Y_i)
    return X, Z, Y

# Construct protocol

Partition chain into two, build GHZ on left and right halves and perform fast state transfer between them

In [ ]:
LL = 10     # multiple of 2

X, Z, Y = generate_Paulis(LL)
I = identity(2**LL, format='csc')

# Step 1: grow GHZ on left and right halves
C_left = I.copy()
C_right = I.copy()
for i in range(1,LL//2):
    C_left = C_left * (I+Z[0] + X[i]*(I-Z[0]))/2
for i in range(LL//2,LL-1):
    C_right = C_right * (I+Z[-1] + X[i]*(I-Z[-1]))/2
U1 = C_left * C_right * (X[-1]+Z[-1])/np.sqrt(2)
# Step 2: entangle GHZs
H2 = csc_matrix((2**LL,2**LL))
for i in range(LL//2):
    for j in range(LL//2,LL):
        H2 += (I-Z[i])*(I-Z[j])
t2 = np.pi/4/(LL//2)**2
U2 = expm(-1j*H2*t2)
# Step 3: Hadamard rotate right GHZ
U3 = C_right * (X[-1]+Z[-1])/np.sqrt(2) * C_right
# Step 4: Hadamard rotate left GHZ
U4 = C_left * (X[0]+Z[0])/np.sqrt(2) * C_left
# Step 5: disentangle GHZs
U5 = U2.T.conj()
# Step 6: shrink right GHZ to rightmost site
U6 = C_right
# Combine all steps
U_ghz = U6 * U5 * U4 * U3 * U2 * U1

### Check that state transfer succeeded on some initial state

In [ ]:
rho0 = csc_matrix([[0.36,0.48],[0.48,0.64]])    # arbitrary initial pure state
for i in range(LL-1):
    rho0 = kron(rho0, csc_matrix([[1,0],[0,0]]), format='csc')
print('Initial expvals:', (rho0*Z[0]).diagonal().sum(), (rho0*X[0]).diagonal().sum())
#rho = U_ghz * rho0 * U_ghz.T.conj()
Utemp=U3*U2*U1
rho = Utemp * rho0 * Utemp.T.conj()
print('Final expvals:', (rho*Z[-1]).diagonal().sum(), (rho*X[-1]).diagonal().sum())

### Compute commutator spectrum

In [ ]:
X_t = U_ghz.T.conj() * X[-1] * U_ghz
Z_t = U_ghz.T.conj() * Z[-1] * U_ghz
comm_ghz=X_t*Z[0]-Z[0]*X_t
#print('Operator norm:', norm(comm_ghz, ord=2) / 1)
#print('Frobenius norm:', norm(comm_ghz, ord='fro') / (2**(L/2)))
#print('Trace norm:', np.linalg.norm(comm_ghz.toarray(), ord='nuc')/ (2**(L)))

In [ ]:
comm_ghz_evals=np.linalg.eigvals(comm_ghz.toarray())

In [ ]:
p_list=np.concatenate([np.linspace(2,50,49,dtype='int'),np.linspace(50,1000,96,dtype='int')])
p_norm_ghz=[]
for p in p_list:
    p_norm_ghz += [np.linalg.norm(abs(comm_ghz_evals),ord=p)/(2**(LL/p))]

# Construct Symmetric Protocol

In [ ]:
L = 10     #multiple of 3, plus 1

X, Z, Y = generate_Paulis(L)
I = identity(2**L, format='csc')

#Step 0: reset symmetrized ancillas
#Basis for 3-qubit symmetric subspace
ket0=csc_matrix([1,0])
ket1=csc_matrix([0,1])

e1=kron(kron(ket0,ket0,format='csc'),ket0,format='csc')
e2=kron(kron(ket1,ket1,format='csc'),ket1,format='csc')
e3=kron(kron(ket1,ket0,format='csc'),ket0,format='csc')+kron(kron(ket0,ket1,format='csc'),ket0,format='csc')+kron(kron(ket0,ket0,format='csc'),ket1,format='csc')
e3=e3/np.sqrt(3)
e4=kron(kron(ket1,ket1,format='csc'),ket0,format='csc')+kron(kron(ket1,ket0,format='csc'),ket1,format='csc')+kron(kron(ket0,ket1,format='csc'),ket1,format='csc')
e4=e4/np.sqrt(3)
P_symm = e1.T@e1 + e2.T@e2 + e3.T@e3 + e4.T@e4
I3 = identity(2**3, format='csc')

e5=kron(kron(ket1,ket0,format='csc'),ket0,format='csc')-2*kron(kron(ket0,ket1,format='csc'),ket0,format='csc')+kron(kron(ket0,ket0,format='csc'),ket1,format='csc')
e5=e5/np.sqrt(6)
e6=kron(kron(ket1,ket1,format='csc'),ket0,format='csc')-2*kron(kron(ket1,ket0,format='csc'),ket1,format='csc')+kron(kron(ket0,ket1,format='csc'),ket1,format='csc')
e6=e6/np.sqrt(6)
e7=kron(kron(ket0,ket0,format='csc'),ket1,format='csc')-kron(kron(ket1,ket0,format='csc'),ket0,format='csc')
e7=e7/np.sqrt(2)
e8=kron(kron(ket0,ket1,format='csc'),ket1,format='csc')-kron(kron(ket1,ket1,format='csc'),ket0,format='csc')
e8=e8/np.sqrt(2)

Usymm_singlegroup = (kron(kron(ket0,ket0,format='csc'),ket0,format='csc').T@e1.conj()+ 
kron(kron(ket1,ket1,format='csc'),ket0,format='csc').T@e2.conj()+ 
kron(kron(ket1,ket0,format='csc'),ket0,format='csc').T@e3.conj()+ 
kron(kron(ket0,ket1,format='csc'),ket0,format='csc').T@e4.conj()+
kron(kron(ket0,ket0,format='csc'),ket1,format='csc').T@e5.conj()+ 
kron(kron(ket1,ket0,format='csc'),ket1,format='csc').T@e6.conj()+ 
kron(kron(ket0,ket1,format='csc'),ket1,format='csc').T@e7.conj()+ 
kron(kron(ket1,ket1,format='csc'),ket1,format='csc').T@e8.conj())

U0 = I2
for i in range((L-1)//3):
    U0 = kron(U0, Usymm_singlegroup, format='csc')

## Steps modified to only act on one in every three qubits
# Step 1: grow GHZ on left and right thirds
Leff=(L-1)//3+1
C_left = I.copy()
C_right = I.copy()
for i in range(1,Leff//2):
    C_left = C_left * (I+Z[0] + X[3*i]*(I-Z[0]))/2
for i in range(Leff//2,Leff-1):
    C_right = C_right * (I+Z[-1] + X[3*i]*(I-Z[-1]))/2
U1 = C_left * C_right * (X[-1]+Z[-1])/np.sqrt(2)
# Step 2: entangle GHZs
H2 = csc_matrix((2**L,2**L))
for i in range(0,Leff//2):
    for j in range(Leff//2,Leff):
        H2 += (I-Z[3*i])*(I-Z[3*j])
t2 = np.pi/4/(Leff//2)**2
U2 = expm(-1j*H2*t2)
# Step 3: Hadamard rotate right GHZ
U3 = C_right * (X[-1]+Z[-1])/np.sqrt(2) * C_right
# Step 4: Hadamard rotate left GHZ
U4 = C_left * (X[0]+Z[0])/np.sqrt(2) * C_left
# Step 5: disentangle GHZs
U5 = U2.T.conj()
# Step 6: shrink right GHZ to rightmost site
U6 = C_right
# Combine all steps
U_symm_ghz = U6 * U5 * U4 * U3 * U2 * U1 * U0

### Verify Usymm_singlegroup

In [ ]:
rand_angle = np.random.rand()          # random uniform state on ancilla qubits
psi_bulk = csc_matrix([rand_angle, np.sqrt(1-rand_angle**2)]).T
psi3 = kron(kron(psi_bulk,psi_bulk),psi_bulk, format='csc')
psi3t = Usymm_singlegroup * psi3
X3, Z3, Y3 = generate_Paulis(3)
print((psi3t.T.conj()*Z3[2]*psi3t)[0,0])

### Check that state transfer succeeded on some initial state

In [ ]:
psi0 = csc_matrix([0.6,0.8]).T    # arbitrary initial pure state
rand_angle = np.random.rand()          # random uniform state on ancilla qubits
psi_bulk = csc_matrix([rand_angle, np.sqrt(1-rand_angle**2)]).T
for i in range(L-1):
    psi0 = kron(psi0, psi_bulk, format='csc')
print('Initial expvals:', (psi0.T.conj()*Z[0]*psi0)[0,0], (psi0.T.conj()*X[0]*psi0)[0,0])
psi =  U_symm_ghz @ psi0
#psi =  U3*U2*U1*U0 @ psi0
print('Final expvals:', (psi.T.conj()*Z[-1]*psi)[0,0], (psi.T.conj()*X[-1]*psi)[0,0])

### Compute commutator spectrum

In [ ]:
X_t = U_symm_ghz.T.conj() * X[-1] * U_symm_ghz
Z_t = U_symm_ghz.T.conj() * Z[-1] * U_symm_ghz
comm_symm_ghz=X_t*Z[0]-Z[0]*X_t
#print('Operator norm:', norm(comm_ghz, ord=2) / 1)
#print('Frobenius norm:', norm(comm_ghz, ord='fro') / (2**(L/2)))
#print('Trace norm:', np.linalg.norm(comm_ghz.toarray(), ord='nuc')/ (2**(L)))

In [ ]:
csg=comm_symm_ghz.toarray()
comm_symm_ghz__evals=np.linalg.eigvals(csg)

In [ ]:
p_norm_symm_ghz=[]
for p in p_list:
    p_norm_symm_ghz += [np.linalg.norm(abs(comm_symm_ghz__evals),ord=p)/(2**(L/p))]

In [ ]:
import matplotlib as mpl
invp_list=1/np.array(p_list)
plt.semilogx(invp_list,p_norm_ghz,label=r'Fast GHZ',color='blue')
plt.semilogx(invp_list,p_norm_symm_ghz,label=r'Symmetrized Fast GHZ',color='green') 

def f_bound_ghz(p):
    return 2*2**((1-LL)/p)

def f_bound_symm_ghz(p):
    return 2*2**(-(L+1)/3/p)
           
mpl.rcParams.update({
    'font.size': 16,        
    'axes.labelsize': 18,     
    'legend.fontsize':14,
})
plt.tick_params(colors='0.25', labelsize=16)

plt.semilogx(invp_list,f_bound_ghz(np.array(p_list)),linestyle='--',color='blue')
plt.semilogx(invp_list,f_bound_symm_ghz(np.array(p_list)),linestyle='--',color='green')

plt.xlabel(r'$1/p$')
plt.ylabel(r'commutator $p$-norm')

plt.legend()
#plt.savefig('fig.png',format='png',dpi=150,bbox_inches='tight',facecolor='white')